<a href="https://colab.research.google.com/github/jeombagis/Stock-price-prediction/blob/main/Git_Data_extraction_TMY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import yfinance as yf
import pandas as pd
import os
from datetime import datetime

# 1. 구글 드라이브 마운트 (이미 되어있다면 생략 가능)
from google.colab import drive
drive.mount('/content/drive')

# 2. 저장할 경로 설정 (기존과 동일한 경로)
save_dir = '/content/drive/MyDrive/Jeombagis/DongnaeFriends/StockPricePrediction/Internal_Build_Ver/TMY'
os.makedirs(save_dir, exist_ok=True)

# 3. 야후 파이낸스 티커(Ticker) 기호
# ^GSPC = S&P 500 지수 / ^NDX = 나스닥 100 지수
indices = {
    "SnP500": "^GSPC",
    "Nasdaq100": "^NDX"
}

# 4. 데이터 추출 및 CSV 저장
end_date = datetime.now().strftime('%Y-%m-%d')
start_date = "1960-01-01" # 약 60년치

for name, ticker in indices.items():
    print(f"\n📥 {name} ({ticker}) 60년치 데이터 추출 중...")

    # 야후 파이낸스에서 데이터 다운로드
    df = yf.download(ticker, start=start_date, end=end_date, progress=False)

    if df.empty:
        print(f"❌ {name} 데이터를 불러오지 못했습니다.")
        continue

    # yfinance 최신 버전 대응 (멀티인덱스 컬럼 평탄화)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    # 인덱스(날짜)를 일반 컬럼으로 변환
    df = df.reset_index()

    # 컬럼명을 기존 키움증권 데이터(소문자)와 동일하게 변경
    df.rename(columns={
        'Date': 'date',
        'Open': 'open',
        'High': 'high',
        'Low': 'low',
        'Close': 'close',
        'Volume': 'volume'
    }, inplace=True)

    # 필요한 컬럼만 순서대로 선택 (Adj Close 등 불필요한 컬럼 제거)
    df = df[['date', 'open', 'high', 'low', 'close', 'volume']]

    # 결측치 제거
    df = df.dropna()

    # 날짜 포맷을 'YYYY-MM-DD'에서 'YYYYMMDD'로 변경 (기존 데이터와 완벽 통일)
    df['date'] = df['date'].dt.strftime('%Y%m%d')

    # 내림차순 정렬 (최신 날짜가 위로 오게 하려면 ascending=False, 과거가 위면 True)
    # 기존 키움증권 데이터가 최신 날짜가 맨 위에 있으므로 이에 맞춤
    df = df.sort_values(by='date', ascending=False).reset_index(drop=True)

    # 5. CSV 파일로 저장
    file_name = f"{name}_60years_daily.csv"
    save_path = os.path.join(save_dir, file_name)

    df.to_csv(save_path, index=False, encoding="utf-8-sig")
    print(f"🎉 [성공] 총 {len(df)}행 저장 완료! -> {save_path}")

print("\n✅ 모든 미국 지수 데이터 수집이 완료되었습니다!")

Mounted at /content/drive

📥 SnP500 (^GSPC) 60년치 데이터 추출 중...


/tmp/ipykernel_8087/3797028589.py:29: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=start_date, end=end_date, progress=False)


🎉 [성공] 총 16681행 저장 완료! -> /content/drive/MyDrive/Jeombagis/DongnaeFriends/StockPricePrediction/Internal_Build_Ver/TMY/SnP500_60years_daily.csv

📥 Nasdaq100 (^NDX) 60년치 데이터 추출 중...


/tmp/ipykernel_8087/3797028589.py:29: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=start_date, end=end_date, progress=False)


🎉 [성공] 총 10213행 저장 완료! -> /content/drive/MyDrive/Jeombagis/DongnaeFriends/StockPricePrediction/Internal_Build_Ver/TMY/Nasdaq100_60years_daily.csv

✅ 모든 미국 지수 데이터 수집이 완료되었습니다!
